# Python Iterators and Generators Exercises: 30 Coding Problems with Solutions

A practice notebook on the iterator protocol, `yield`, `yield from`, lazy pipelines, coroutines (`.send()`/`.throw()`), and generator cleanup (`.close()`) — each with a concept note, a hint, a solution, and an explanation.

*Adapted for practice from the exercise list at [PYnative](https://pynative.com/python-iterators-generators-exercises/). Each code cell creates any sample file(s)/folder(s) it needs, so the notebook runs standalone from top to bottom.*

---

## Concepts you'll need

This set covers Python's **iterator protocol** and **generators**, from basics through coroutines and cleanup.

- **The iterator protocol** — an *iterable* has `__iter__()`, which returns an *iterator*; an iterator has both `__iter__()` (usually returning `self`) and `__next__()`, which returns the next value or raises `StopIteration` when exhausted. A `for` loop is just `iter()` once, then `next()` repeatedly, catching `StopIteration` — writing this manually (Exercise 9) makes the mechanism explicit.
- **Generator functions** — any function containing `yield` becomes a generator function; calling it doesn't run the body, it returns a generator object. Each call to `next()` resumes execution right after the last `yield`, runs until the next `yield` (or the function ends, which raises `StopIteration`), and local variables persist across pauses.
- **Generator expressions** — `(expr for x in iterable if cond)`, the lazy lambda-free cousin of a list comprehension; nothing is computed until a value is requested.
- **`yield from`** — delegates to a sub-iterable, forwarding every value it produces (and, importantly, forwarding `send()`/`throw()` into it too) — essential for recursive generators (tree/directory traversal, nested flattening) without a manual inner loop.
- **Infinite generators** — a `while True: yield ...` generator never runs forever on its own; it only advances when something calls `next()`, so the caller (often with `break`) controls when to stop.
- **Coroutines: `.send()` and `.throw()`** — `value = yield x` is two-directional: it yields `x` out and receives whatever's passed to `.send()` on the next resume. `.throw(ExcType, msg)` raises that exception *at* the generator's current `yield`, letting a `try/except` inside the generator handle it and keep going.
- **`.close()` and cleanup** — calling `.close()` throws `GeneratorExit` into the generator at its paused point; a `try/finally` wrapped around the yielding code guarantees cleanup (closing a file, a DB connection) runs whether the generator finishes normally, errors, or is closed early — the same guarantee a `with` statement provides.
- **Why any of this matters** — generators produce values one at a time with O(1) memory regardless of sequence length, which is what makes them the right tool for large files, infinite sequences, and composable lazy pipelines where each stage processes one item before passing it to the next.

Each exercise below gives a problem, a hint, a solution, and an explanation.

## Exercise 1. The Square Generator

**Concept:** a basic generator function with yield

**Problem:** Write a generator that yields the squares of numbers from 1 to n.

**Given:**
```
n = 5
```

**Expected Output:**
```
1 4 9 16 25
```

**Hint:** yield produces one value and pauses; the function resumes right after that yield on the next call.

In [ ]:
def square_generator(n):
    for i in range(1, n + 1):
        yield i ** 2

for square in square_generator(5):
    print(square, end=" ")

**Explanation:** Having yield anywhere in a function body is what makes it a generator function rather than a regular one — calling square_generator(5) doesn't run any code yet, it just returns a generator object. Each pass of the for loop inside computes one square and yield pauses execution there, handing that value to the caller; range(1, n + 1) ensures n itself is included, since range's upper bound is exclusive.

## Exercise 2. Even Number Iterator

**Concept:** a class-based iterator with __iter__ and __next__

**Problem:** Build a class-based iterator that returns even numbers up to a limit.

**Given:**
```
limit = 10
```

**Expected Output:**
```
0 2 4 6 8 10
```

**Hint:** __iter__ returning self is what satisfies Python's requirement that an iterable produce an iterator when iter() is called.

In [ ]:
class EvenIterator:
    def __init__(self, limit):
        self.limit = limit
        self.current = 0

    def __iter__(self):
        return self

    def __next__(self):
        if self.current > self.limit:
            raise StopIteration
        value = self.current
        self.current += 2
        return value

for num in EvenIterator(10):
    print(num, end=" ")

**Explanation:** self.current tracks the next value to return, starting at 0 and advancing by 2 every call. __next__ is what for loops (and other iteration constructs) call automatically, and raising StopIteration once current exceeds limit is the standard signal that tells the loop to stop cleanly. Unlike a generator function, this class approach gives full manual control over the internal state, at the cost of more boilerplate.

## Exercise 3. Custom Range

**Concept:** reimplementing range() as a generator

**Problem:** Re-implement a simplified version of range() using a generator.

**Given:**
```
start = 2, stop = 10, step = 2
```

**Expected Output:**
```
2 4 6 8
```

**Hint:** while current < stop excludes the stop value, matching the built-in range()'s behavior.

In [ ]:
def custom_range(start, stop, step=1):
    current = start
    while current < stop:
        yield current
        current += step

for num in custom_range(2, 10, 2):
    print(num, end=" ")

**Explanation:** current = start is initialized once, outside the loop, so it persists across every yield since it lives in the generator's own local scope. The while current < stop condition excludes the stop boundary itself, exactly mirroring the built-in range(). Placing current += step AFTER yield ensures the current (not yet incremented) value is what actually gets produced each time.

## Exercise 4. Reverse String Iterator

**Concept:** a class-based iterator walking a string backwards

**Problem:** Build an iterator that returns a string's characters in reverse order.

**Given:**
```
text = "hello"
```

**Expected Output:**
```
o l l e h
```

**Hint:** Starting index at len(text) - 1 points to the last valid character position.

In [ ]:
class ReverseStringIterator:
    def __init__(self, text):
        self.text = text
        self.index = len(text) - 1

    def __iter__(self):
        return self

    def __next__(self):
        if self.index < 0:
            raise StopIteration
        char = self.text[self.index]
        self.index -= 1
        return char

for char in ReverseStringIterator("hello"):
    print(char, end=" ")

**Explanation:** len(text) - 1 gives the last valid index — for 'hello' (length 5) that's index 4, pointing at 'o'. Each call to __next__ grabs the character at the current index BEFORE decrementing, ensuring the right character is returned; once index drops below 0, every character has been visited and StopIteration ends the iteration.

## Exercise 5. Vowel Filter

**Concept:** the generator-as-filter pattern

**Problem:** Write a generator that yields only the vowels from a string.

**Given:**
```
text = "Hello, World!"
```

**Expected Output:**
```
e o o
```

**Hint:** char.lower() in vowels handles both uppercase and lowercase input with one membership check.

In [ ]:
def vowel_filter(text):
    vowels = "aeiou"
    for char in text:
        if char.lower() in vowels:
            yield char

for vowel in vowel_filter("Hello, World!"):
    print(vowel, end=" ")

**Explanation:** Lowercasing each character before the membership check means both 'H' and 'h' are handled correctly without needing to list uppercase vowels separately. yield char is placed only inside the if block, so non-vowel characters are simply skipped entirely rather than yielded and filtered later — this generator-as-filter pattern processes the string one character at a time with no intermediate list ever built.

## Exercise 6. Power of Two

**Concept:** a generator expression — the lazy one-line alternative to a generator function

**Problem:** Write a one-line generator expression that yields powers of 2.

**Given:**
```
n = 8
```

**Expected Output:**
```
1 2 4 8 16 32 64 128
```

**Hint:** Parentheses (not square brackets) make it a lazy generator expression instead of an eager list comprehension.

In [ ]:
n = 8
powers = (2 ** i for i in range(n))

for value in powers:
    print(value, end=" ")

**Explanation:** (2 ** i for i in range(n)) behaves exactly like a generator function with yield, just condensed into a single expression — no list is ever built in memory. range(n) produces exponents 0 through n-1, and since 2**0 = 1, the sequence correctly starts there. Swapping the outer parentheses for square brackets would instead build the entire list immediately; the generator version defers all computation until the for loop actually asks for each value.

## Exercise 7. Finite Fibonacci

**Concept:** a generator carrying two pieces of state across yields

**Problem:** Write a generator producing the first n Fibonacci numbers.

**Given:**
```
n = 8
```

**Expected Output:**
```
0 1 1 2 3 5 8 13
```

**Hint:** a, b = b, a + b advances both values in one step, using the OLD value of a to compute the new b.

In [ ]:
def fibonacci(n):
    a, b = 0, 1
    for _ in range(n):
        yield a
        a, b = b, a + b

for num in fibonacci(8):
    print(num, end=" ")

**Explanation:** a, b = 0, 1 seeds the sequence, and both variables persist in the generator's local scope across every yield. yield a produces the current term and pauses; the following a, b = b, a + b advances the sequence — Python evaluates the entire right-hand side (b and a + b) before assigning either variable, so the old a is correctly used to compute the new b. for _ in range(n) simply bounds the output to exactly n terms.

## Exercise 8. Infinite Counter

**Concept:** an unbounded generator, safely consumed with break

**Problem:** Create a generator that counts up forever, consumed safely with a break condition.

**Given:**
```
print values until the counter passes 5
```

**Expected Output:**
```
1 2 3 4 5
```

**Hint:** while True inside a generator is safe precisely because execution pauses at yield each time — it never actually runs unboundedly on its own.

In [ ]:
def infinite_counter(start=1):
    current = start
    while True:
        yield current
        current += 1

for num in infinite_counter():
    if num > 5:
        break
    print(num, end=" ")

**Explanation:** while True creates a loop with no natural end inside the generator, but this is safe: because execution pauses at yield, the loop body only actually advances when the CALLER requests another value via next(). The consuming for loop is what imposes the actual stopping point — without the if num > 5: break guard, this would genuinely run forever, since the generator itself has no built-in limit.

## Exercise 9. Manual Iteration

**Concept:** iter() + next() — what a for loop does internally

**Problem:** Manually iterate a list using iter() and next(), catching StopIteration yourself.

**Given:**
```
numbers = [10, 20, 30]
```

**Expected Output:**
```
10 20 30
```

**Hint:** This pattern — iter() once, then next() repeatedly until StopIteration — is exactly what a for loop does under the hood.

In [ ]:
numbers = [10, 20, 30]
iterator = iter(numbers)

while True:
    try:
        value = next(iterator)
        print(value, end=" ")
    except StopIteration:
        break

**Explanation:** iter(numbers) calls numbers.__iter__() internally and returns an iterator — lists are iterable, but are not themselves iterators, which is exactly why this separate step is needed. Each next(iterator) call advances the internal position by one and returns the value there; once the list is exhausted, next() raises StopIteration, which this code catches directly and uses to break the loop — precisely mirroring what a for loop does automatically and invisibly.

## Exercise 10. Step Iterator

**Concept:** extending the custom-range idea to support float steps

**Problem:** Build a class-based iterator supporting float steps, which the built-in range() cannot do.

**Given:**
```
start = 0.0, stop = 1.0, step = 0.25
```

**Expected Output:**
```
0.0 0.25 0.5 0.75
```

**Hint:** round(..., 10) corrects tiny floating-point accumulation errors without affecting normal step-size precision.

In [ ]:
class FloatStepIterator:
    def __init__(self, start, stop, step):
        self.current = start
        self.stop = stop
        self.step = step

    def __iter__(self):
        return self

    def __next__(self):
        if self.current >= self.stop:
            raise StopIteration
        value = self.current
        self.current = round(self.current + self.step, 10)
        return value

for num in FloatStepIterator(0.0, 1.0, 0.25):
    print(num, end=" ")

**Explanation:** Python's built-in range() only accepts integers — this class fills that specific gap by using start directly as a float-friendly running position. current >= stop ends iteration once the boundary is reached or passed, matching range()'s exclusive upper bound. Floating-point addition can accumulate tiny errors (like 0.1 + 0.2 producing 0.30000000000000004), so rounding to 10 decimal places after each step corrects that drift without meaningfully affecting precision at normal step sizes.

## Exercise 11. File Line Reader

**Concept:** memory-efficient file reading via a generator

**Problem:** Write a generator that reads a text file line by line without loading it all into memory.

**Given:**
```
sample.txt with 3 lines
```

**Expected Output:**
```
Hello, World!
Python is great.
Generators save memory.
```

**Hint:** File objects are themselves iterable — a plain for loop over one already reads lazily, one line at a time.

In [ ]:
with open("sample.txt", "w") as f:
    f.write("Hello, World!\nPython is great.\nGenerators save memory.\n")

def read_lines(filepath):
    with open(filepath, "r") as f:
        for line in f:
            yield line.rstrip("\n")

for line in read_lines("sample.txt"):
    print(line)

**Explanation:** The with block inside the generator guarantees the file closes automatically, even if the caller stops iterating partway through. File objects implement the iterator protocol natively — for line in f reads exactly one line from disk per iteration, so only a single line ever occupies memory at once, regardless of whether the file is 1 KB or 10 GB. line.rstrip("\n") strips the trailing newline each line carries so the printed output doesn't show blank gaps between lines.

## Exercise 12. CSV Row Parser

**Concept:** combining next() (to consume the header) with zip() and dict()

**Problem:** Write a generator that yields CSV rows as dictionaries, using the header row as keys.

**Given:**
```
data.csv with a header row plus 3 data rows
```

**Expected Output:**
```
{'name': 'Alice', 'age': '30', 'city': 'New York'} ...
```

**Hint:** Calling next(f) once BEFORE the for loop consumes just the header line, leaving only data rows for the loop.

In [ ]:
with open("data.csv", "w") as f:
    f.write("name,age,city\nAlice,30,New York\nBob,25,London\nCarol,35,Sydney\n")

def csv_row_parser(filepath):
    with open(filepath, "r") as f:
        headers = next(f).strip().split(",")
        for line in f:
            values = line.strip().split(",")
            yield dict(zip(headers, values))

for row in csv_row_parser("data.csv"):
    print(row)

**Explanation:** next(f) advances the file's own iterator by exactly one line and returns it — calling this before the for loop consumes the header row, so the loop itself only ever sees data rows. zip(headers, values) pairs each column name with its corresponding value by position, and dict() builds the row dictionary from those pairs in one step. .strip() removes the trailing newline before .split(",") breaks each line into fields — skipping strip() would leave a stray \n stuck to each row's last value. (Python's built-in csv.DictReader handles real-world edge cases like quoted commas that this simplified version doesn't.)

## Exercise 13. The Pipeline

**Concept:** composing generators — output of one feeds directly into the next

**Problem:** Build two generators, one producing values and one squaring them, and chain them together.

**Given:**
```
numbers = [1, 2, 3, 4, 5]
```

**Expected Output:**
```
1 4 9 16 25
```

**Hint:** squarer(number_producer(numbers)) composes the two generators without running either — nothing executes until the final for loop pulls values.

In [ ]:
def number_producer(iterable):
    for item in iterable:
        yield item

def squarer(iterable):
    for num in iterable:
        yield num ** 2

numbers = [1, 2, 3, 4, 5]
pipeline = squarer(number_producer(numbers))

for value in pipeline:
    print(value, end=" ")

**Explanation:** number_producer simply passes each item through unchanged — in a real pipeline this stage would typically filter or transform. squarer accepts any iterable (including another generator) and squares each value, making it decoupled from where its input actually comes from. squarer(number_producer(numbers)) is lazy composition: neither generator's code runs at that line — only when the final for loop calls next() on the composed pipeline does computation actually happen, one value at a time through the whole chain.

## Exercise 14. List Flattener

**Concept:** yield from + recursion for arbitrary-depth flattening

**Problem:** Use yield from to build a recursive generator that flattens a nested list.

**Given:**
```
nested = [1, [2, [3, 4], 5], [6, 7], 8]
```

**Expected Output:**
```
1 2 3 4 5 6 7 8
```

**Hint:** yield from transparently forwards every value the recursive call produces, with no manual inner loop needed.

In [ ]:
def flatten(nested):
    for item in nested:
        if isinstance(item, list):
            yield from flatten(item)
        else:
            yield item

nested = [1, [2, [3, 4], 5], [6, 7], 8]

for value in flatten(nested):
    print(value, end=" ")

**Explanation:** isinstance(item, list) is the branch point: if the current item is itself a list, the function recurses into it via yield from flatten(item); otherwise it's a plain value, yielded directly. yield from delegates to the recursive call, forwarding every value it eventually yields straight through to the outermost caller — without it you'd need `for v in flatten(item): yield v`, and yield from is not just shorter, it also correctly forwards send()/throw() into the sub-generator, which the manual loop cannot do. Because each nested list triggers its own recursive call, arbitrary nesting depth is handled with the exact same code.

## Exercise 15. Batch Processing

**Concept:** grouping a stream into fixed-size chunks with safe slicing

**Problem:** Write a generator that yields fixed-size chunks of an iterable.

**Given:**
```
items = list(range(1, 11)), batch_size = 3
```

**Expected Output:**
```
[1, 2, 3]
[4, 5, 6]
[7, 8, 9]
[10]
```

**Hint:** Python slicing never raises IndexError for an out-of-range upper bound — the final chunk is just shorter automatically.

In [ ]:
def batch(iterable, batch_size):
    items = list(iterable)
    for i in range(0, len(items), batch_size):
        yield items[i : i + batch_size]

for chunk in batch(range(1, 11), 3):
    print(chunk)

**Explanation:** items = list(iterable) materializes the input so it can be sliced by index — necessary because generators and other one-shot iterables don't support slicing directly. range(0, len(items), batch_size) produces starting indices 0, 3, 6, 9, each marking the start of one chunk. items[i : i + batch_size] never raises an error even when i + batch_size overruns the list's length, which is exactly why the final chunk ([10]) comes back shorter with no special-case handling required.

## Exercise 16. Prime Sieve

**Concept:** the Sieve of Eratosthenes expressed as a lazy generator

**Problem:** Implement a generator yielding primes using sieve logic rather than trial division.

**Given:**
```
limit = 30
```

**Expected Output:**
```
2 3 5 7 11 13 17 19 23 29
```

**Hint:** Checking factors only up to sqrt(limit) is enough, since any composite has a factor at or below its own square root.

In [ ]:
def prime_sieve(limit):
    is_prime = [True] * (limit + 1)
    is_prime[0] = is_prime[1] = False

    for i in range(2, int(limit ** 0.5) + 1):
        if is_prime[i]:
            is_prime[i * i :: i] = [False] * len(is_prime[i * i :: i])

    for i in range(2, limit + 1):
        if is_prime[i]:
            yield i

for prime in prime_sieve(30):
    print(prime, end=" ")

**Explanation:** is_prime starts as a boolean lookup table where every index is assumed prime; the sieve progressively marks composites False. The outer loop only needs to run up to sqrt(limit), since any composite number must have a factor at or below its own square root — by that point all composites are guaranteed marked. The slice assignment is_prime[i*i::i] = [False] * ... marks every multiple of i starting from i-squared (smaller multiples like 2i and 3i were already marked by earlier, smaller primes) in one operation. Only after the whole table is built does the second loop yield the surviving primes — a deliberate two-phase design trading a bit of latency for correctness.

## Exercise 17. Running Average

**Concept:** a stateful generator tracking a running total and count

**Problem:** Write a generator that yields the running average of a stream of numbers.

**Given:**
```
numbers = [10, 20, 30, 40, 50]
```

**Expected Output:**
```
10.0 15.0 20.0 25.0 30.0
```

**Hint:** Updating total and count BEFORE yielding ensures the average includes the current value, not a one-step-behind one.

In [ ]:
def running_average(iterable):
    total = 0
    count = 0
    for value in iterable:
        total += value
        count += 1
        yield total / count

numbers = [10, 20, 30, 40, 50]

for avg in running_average(numbers):
    print(avg, end=" ")

**Explanation:** total and count are declared before the loop and, because they live in the generator's own local scope, persist across every yield, accumulating state one value at a time. Both are updated BEFORE the yield, so each yielded average correctly reflects the just-processed value — updating after yield instead would produce an average that's always one step behind. Python 3's / always returns a float, which is why the output shows 10.0 rather than plain 10 even for whole-number results. Only two scalar variables are ever kept in memory, regardless of how many numbers have streamed through — no history of past values is stored at all.

## Exercise 18. Sliding Window

**Concept:** yielding overlapping fixed-size windows over a sequence

**Problem:** Write a generator that yields a sliding window of size n over a sequence.

**Given:**
```
sequence = [1, 2, 3, 4, 5], n = 3
```

**Expected Output:**
```
(1, 2, 3) (2, 3, 4) (3, 4, 5)
```

**Hint:** range(len(items) - n + 1) computes exactly how many full windows of size n fit.

In [ ]:
def sliding_window(sequence, n):
    items = list(sequence)
    for i in range(len(items) - n + 1):
        yield tuple(items[i : i + n])

sequence = [1, 2, 3, 4, 5]

for window in sliding_window(sequence, 3):
    print(window, end=" ")

**Explanation:** items = list(sequence) materializes the input into an indexable list, which matters because a one-shot iterable passed as sequence would otherwise be exhausted after a single access. range(len(items) - n + 1) computes exactly how many complete windows fit — for 5 items and a window of 3, that's range(3), giving starting indices 0, 1, 2 for the three valid windows. Wrapping each slice in tuple() marks the window as a fixed-size, ordered snapshot rather than a mutable list, the conventional shape for windowed data. (Python 3.12+ offers itertools.sliding_window() as a built-in equivalent.)

## Exercise 19. Log Filter

**Concept:** file reading + conditional filtering combined in one generator

**Problem:** Write a generator that yields only log lines containing a specific keyword.

**Given:**
```
app.log with a mix of INFO/ERROR/WARNING lines
```

**Expected Output:**
```
ERROR: Disk quota exceeded
ERROR: Database connection failed
```

**Hint:** A default keyword parameter makes this generator reusable for any log level, not just ERROR.

In [ ]:
log_content = (
    "INFO: Server started\n"
    "ERROR: Disk quota exceeded\n"
    "INFO: Request received\n"
    "ERROR: Database connection failed\n"
    "WARNING: High memory usage\n"
)
with open("app.log", "w") as f:
    f.write(log_content)

def error_filter(filepath, keyword="ERROR"):
    with open(filepath, "r") as f:
        for line in f:
            line = line.strip()
            if keyword in line:
                yield line

for entry in error_filter("app.log"):
    print(entry)

**Explanation:** keyword="ERROR" as a default parameter makes the generator immediately reusable — passing keyword="WARNING" filters for warnings instead, with zero code changes needed. line.strip() removes both the trailing newline and any surrounding whitespace before the membership check runs, and before yielding, so printed output has no stray blank lines. if keyword in line is a plain substring check — this is the same generator-as-filter pattern from Exercise 5, just applied to lines read from a file instead of characters in a string. Because the output is individual strings, this generator can feed straight into another one (say, extracting timestamps) with no changes to this stage.

## Exercise 20. Unique Element Filter

**Concept:** order-preserving deduplication via a generator + a hidden set

**Problem:** Write a generator that yields only unique elements, in their original order of first appearance.

**Given:**
```
items = [3, 1, 4, 1, 5, 9, 2, 6, 5, 3, 5]
```

**Expected Output:**
```
3 1 4 5 9 2 6
```

**Hint:** A set (not a list) for tracking 'seen' values keeps membership checks O(1) instead of O(n).

In [ ]:
def unique(iterable):
    seen = set()
    for item in iterable:
        if item not in seen:
            seen.add(item)
            yield item

items = [3, 1, 4, 1, 5, 9, 2, 6, 5, 3, 5]

for value in unique(items):
    print(value, end=" ")

**Explanation:** seen starts empty and persists across yields in the generator's local scope, accumulating every value produced so far. Checking `item not in seen` against a set runs in O(1) average time thanks to hashing; checking the same condition against a plain list would be O(n) per item, making the whole algorithm O(n squared) for large inputs. The item is added to seen BEFORE being yielded, ensuring correctness even if the caller stops early and later resumes iteration. Unlike converting straight to a set() (which discards insertion order entirely), this generator yields values in the exact order they first appeared.

## Exercise 21. The Accumulator (send)

**Concept:** a coroutine using two-directional yield with .send()

**Problem:** Create a generator/coroutine that receives values via .send() and keeps a running total.

**Given:**
```
send 10, then 20, then 30
```

**Expected Output:**
```
10 30 60
```

**Hint:** A generator must be 'primed' with next() (advancing it to its first yield) before .send() can push in real values.

In [ ]:
def accumulator():
    total = 0
    while True:
        value = yield total
        if value is None:
            break
        total += value

acc = accumulator()
next(acc)  # prime the coroutine

print(acc.send(10), end=" ")  # 10
print(acc.send(20), end=" ")  # 30
print(acc.send(30), end=" ")  # 60

**Explanation:** value = yield total is two-directional: the expression to the right of yield (total) is what gets sent OUT to the caller, while whatever is passed into the next .send() call is assigned to value on the LEFT once the generator resumes. next(acc) primes the coroutine by advancing it to its very first yield — at that point total is still 0, and this initial return value is simply discarded since no real data has been sent yet. Each subsequent acc.send(10) resumes the generator, assigns 10 to value, adds it into total, loops back to yield total again, and returns that new running total to the caller. Sending None (via a bare next() call after priming) triggers the break guard, allowing the coroutine to shut down cleanly.

## Exercise 22. Custom Zip

**Concept:** reimplementing zip() to understand StopIteration-as-exit-signal

**Problem:** Re-implement zip() using iter() and next() to combine two iterables.

**Given:**
```
a = [1, 2, 3], b = ["a", "b", "c"]
```

**Expected Output:**
```
(1, 'a') (2, 'b') (3, 'c')
```

**Hint:** Calling return inside a generator is the idiomatic way to end it early — Python converts that return into a StopIteration for the caller.

In [ ]:
def custom_zip(iterable_a, iterable_b):
    iter_a = iter(iterable_a)
    iter_b = iter(iterable_b)
    while True:
        try:
            item_a = next(iter_a)
            item_b = next(iter_b)
        except StopIteration:
            return
        yield (item_a, item_b)

a = [1, 2, 3]
b = ["a", "b", "c"]

for pair in custom_zip(a, b):
    print(pair, end=" ")

**Explanation:** Converting both inputs to iterators with iter() means the function accepts ANY iterable — lists, generators, strings, custom classes — not just things that support indexing. When either next() call raises StopIteration (because one iterator is exhausted), the except clause catches it and return exits the generator immediately — Python automatically converts that return into the StopIteration the calling for loop expects. The yield only happens after BOTH next() calls succeed, which guarantees a partial tuple is never emitted if the second iterable runs out first — exactly the same shortest-iterable-wins behavior as the built-in zip().

## Exercise 23. Binary Tree Traversal

**Concept:** yield from applied to a recursive tree structure

**Problem:** Implement in-order BST traversal using a recursive generator.

**Given:**
```
a BST built from inserting 5, 3, 7, 2, 4, 6, 8
```

**Expected Output:**
```
2 3 4 5 6 7 8
```

**Hint:** For a BST, in-order traversal (left, node, right) always visits values in ascending sorted order.

In [ ]:
class Node:
    def __init__(self, value):
        self.value = value
        self.left = None
        self.right = None

    def insert(self, value):
        if value < self.value:
            if self.left is None:
                self.left = Node(value)
            else:
                self.left.insert(value)
        else:
            if self.right is None:
                self.right = Node(value)
            else:
                self.right.insert(value)

def inorder(node):
    if node is None:
        return
    yield from inorder(node.left)
    yield node.value
    yield from inorder(node.right)

root = Node(5)
for val in [3, 7, 2, 4, 6, 8]:
    root.insert(val)

for value in inorder(root):
    print(value, end=" ")

**Explanation:** if node is None: return is the recursion's base case — a plain return inside a generator raises StopIteration silently, so a leaf's None children simply end that branch's contribution with no visible effect. yield from inorder(node.left) delegates every value from the left subtree straight to the outermost caller, avoiding a manual loop that would otherwise be needed to re-yield each one. The structure — left subtree, then the node's own value, then right subtree — is exactly what makes in-order traversal of a BST always produce values in ascending sorted order. Unlike a list-based approach (O(n) extra memory to collect everything first), this generator only ever needs stack depth proportional to the tree's height.

## Exercise 24. Data Throttler

**Concept:** downsampling a stream by wrapping it with a filtering generator

**Problem:** Create a generator that wraps an iterable and yields only every n-th item.

**Given:**
```
data = list(range(1, 21)), n = 4
```

**Expected Output:**
```
4 8 12 16 20
```

**Hint:** Starting enumerate at 1 (not the default 0) makes the modulo check land on the 4th/8th/12th items, not the 3rd/7th/11th.

In [ ]:
def throttle(iterable, n):
    for index, item in enumerate(iterable, start=1):
        if index % n == 0:
            yield item

data = list(range(1, 21))

for value in throttle(data, 4):
    print(value, end=" ")

**Explanation:** enumerate(iterable, start=1) pairs each item with a counter beginning at 1 rather than 0, which is exactly what makes index % n == 0 correctly identify the true 4th, 8th, 12th positions instead of being off by one. Because throttle accepts any iterable, it can wrap a live sensor feed, a file reader, or another generator just as easily as a plain list — and the upstream iterable is never fully materialized into memory. (itertools.islice(iterable, n-1, None, n) achieves the same downsampling using the standard library.)

## Exercise 25. Generator State Machine

**Concept:** encoding a cyclic state machine as an infinite generator

**Problem:** Model a traffic light that cycles through Green, Yellow, Red using a generator.

**Given:**
```
advance through 6 transitions
```

**Expected Output:**
```
Green Yellow Red Green Yellow Red
```

**Hint:** while True wraps the inner for loop, restarting it immediately after the last state to produce an endless cycle.

In [ ]:
def traffic_light():
    states = ["Green", "Yellow", "Red"]
    while True:
        for state in states:
            yield state

light = traffic_light()

for _ in range(6):
    print(next(light), end=" ")

**Explanation:** states defines the ordered transition sequence — changing this one list is all it would take to alter the machine's behavior, with no logic elsewhere needing to change. The inner for loop cycles through all three states once; the outer while True immediately restarts that cycle the moment it finishes, producing an endless loop the generator never naturally exits. yield state pauses at each state and resumes from exactly that point on the next call, and for _ in range(6): next(light) is what actually decides how many transitions to pull — the generator itself has no idea when it will be stopped, which is the clean separation that makes this pattern reusable.

## Exercise 26. Peekable Iterator

**Concept:** wrapping an iterator with lookahead, using a sentinel to detect exhaustion

**Problem:** Wrap an iterator in a class that lets you peek at the next value without consuming it.

**Given:**
```
data = [10, 20, 30, 40]
```

**Expected Output:**
```
Peek: 10, Consumed: 10, Peek: 20, Consumed: 20, Consumed: 30, Consumed: 40
```

**Hint:** A unique sentinel object (not None) safely distinguishes 'no value cached' from a legitimately cached None or 0.

In [ ]:
_SENTINEL = object()

class PeekableIterator:
    def __init__(self, iterable):
        self._iterator = iter(iterable)
        self._next = next(self._iterator, _SENTINEL)

    def __iter__(self):
        return self

    def peek(self):
        if self._next is _SENTINEL:
            raise StopIteration
        return self._next

    def __next__(self):
        if self._next is _SENTINEL:
            raise StopIteration
        value = self._next
        self._next = next(self._iterator, _SENTINEL)
        return value

data = [10, 20, 30, 40]
pit = PeekableIterator(data)

print("Peek:", pit.peek())
print("Consumed:", next(pit))
print("Peek:", pit.peek())
print("Consumed:", next(pit))
print("Consumed:", next(pit))
print("Consumed:", next(pit))

**Explanation:** _SENTINEL = object() creates a genuinely unique object that could never legitimately appear in a real data stream, so comparing with `is _SENTINEL` stays safe even if the iterable actually contains None, 0, or False — values that would wrongly trigger a naive None-based sentinel. next(self._iterator, _SENTINEL) is the two-argument form of next(), which returns the given default instead of raising StopIteration when exhausted, keeping the caching logic free of try/except. peek() simply returns the cached _next without touching the underlying iterator, so calling it repeatedly between __next__ calls always gives the same answer. __next__ immediately re-primes _next with the following item right after saving the current one, which is exactly what keeps peek() perpetually accurate.

## Exercise 27. Exception Handler

**Concept:** injecting an exception into a paused generator with .throw()

**Problem:** Create a generator that catches a specific exception injected mid-stream via .throw().

**Given:**
```
values 1, 2, 3, with a ValueError injected after the second value
```

**Expected Output:**
```
Yielded: 1
Yielded: 2
Caught ValueError: simulated error - skipping
Yielded: 3
```

**Hint:** Wrapping the yield itself in try/except ValueError is what makes the generator able to survive .throw() and keep going.

In [ ]:
def resilient_generator(values):
    for value in values:
        try:
            yield value
        except ValueError as e:
            print(f"Caught ValueError: {e} - skipping")

gen = resilient_generator([1, 2, 3])

print("Yielded:", next(gen))
print("Yielded:", next(gen))
print("Yielded:", gen.throw(ValueError("simulated error")))

**Explanation:** Wrapping yield inside try/except ValueError is the key detail — without it, a thrown exception would propagate straight out uncaught, terminating the generator immediately. gen.throw(ValueError("simulated error")) raises that exact exception at the precise line the generator is currently paused on (inside the try block), and the generator's own except clause handles it, prints the message, then the for loop simply continues on to yield 3. Just like next(), .throw() returns whatever the generator yields next after handling the exception — which is why printing its return value directly shows 3 with no separate next() call needed. (This same mechanism is what powers task cancellation inside async frameworks like asyncio, via an internally-thrown CancelledError. Passing a single exception instance to .throw(), as done here, is the modern, non-deprecated form — older code sometimes uses the 3-argument gen.throw(ExcType, value, tb) form instead.)

## Exercise 28. Recursive Directory Walker

**Concept:** yield from applied to real filesystem recursion

**Problem:** Write a generator that recursively yields every file path in a directory tree.

**Given:**
```
a test directory tree with nested files, created at runtime
```

**Expected Output:**
```
test_dir/file1.txt
test_dir/file2.txt
test_dir/subdir/file3.txt
test_dir/subdir/file4.txt
```

**Hint:** sorted(os.listdir(...)) matters — filesystem order isn't guaranteed to be alphabetical on its own.

In [ ]:
import os

os.makedirs("test_dir/subdir", exist_ok=True)
for filename in ["test_dir/file1.txt", "test_dir/file2.txt",
                  "test_dir/subdir/file3.txt", "test_dir/subdir/file4.txt"]:
    open(filename, "w").close()

def walk_files(directory):
    for entry in sorted(os.listdir(directory)):
        full_path = os.path.join(directory, entry)
        if os.path.isfile(full_path):
            yield full_path
        elif os.path.isdir(full_path):
            yield from walk_files(full_path)

for path in walk_files("test_dir"):
    print(path)

**Explanation:** os.path.join(directory, entry) builds a platform-correct path — string-concatenating instead would break on Windows, where the separator is a backslash rather than a forward slash. Files are yielded directly, but hitting a directory triggers a recursive call via yield from, which forwards every path the deeper call eventually produces straight up to the outermost caller with no intermediate list. sorted(os.listdir(...)) matters because raw filesystem listing order isn't guaranteed alphabetical, and wrapping it gives consistent, predictable output regardless of OS. (In production code, os.walk() or pathlib.Path.rglob('*') would typically replace this hand-rolled version, but writing it out makes the yield-from recursion pattern explicit.)

## Exercise 29. Interleaved Streams

**Concept:** alternating between two iterators, gracefully handling unequal length

**Problem:** Write a generator that yields elements from two iterables alternately.

**Given:**
```
a = [1, 2, 3], b = ["a", "b", "c", "d"]
```

**Expected Output:**
```
1 a 2 b 3 c d
```

**Hint:** Two SEPARATE try/except blocks (one per iterator) catch exhaustion at the exact moment it happens, for either stream.

In [ ]:
def interleave(iterable_a, iterable_b):
    iter_a = iter(iterable_a)
    iter_b = iter(iterable_b)
    while True:
        try:
            yield next(iter_a)
        except StopIteration:
            yield from iter_b
            return
        try:
            yield next(iter_b)
        except StopIteration:
            yield from iter_a
            return

a = [1, 2, 3]
b = ["a", "b", "c", "d"]

for value in interleave(a, b):
    print(value, end=" ")

**Explanation:** Each iterator gets its own independent try/except block, which catches exhaustion at the precise moment it happens — right after that stream's last successfully-yielded item — rather than at the top of the next loop pass, which could otherwise silently drop an item from the other stream. Once iter_a runs dry, yield from iter_b drains everything remaining in b in one step, and the immediate return prevents the loop from looping back and trying to read the already-exhausted iter_a again. Because this drain-and-return logic lives independently inside each except block, all three length scenarios — equal, a shorter, or b shorter — are handled correctly with no explicit length check needed up front.

## Exercise 30. Generator Cleanup

**Concept:** try/finally inside a generator, cooperating with .close()

**Problem:** Use try/finally in a generator to guarantee a resource is released even if the generator is closed early.

**Given:**
```
rows = [("Alice", 30), ("Bob", 25), ("Carol", 35)], closed after reading only the first row
```

**Expected Output:**
```
[DB] Connection opened
Row: ('Alice', 30)
[DB] Connection closed
```

**Hint:** gen.close() throws a GeneratorExit into the paused generator, which is exactly what triggers the finally block to run.

In [ ]:
class FakeDatabase:
    def __init__(self, rows):
        self.rows = rows

    def connect(self):
        print("[DB] Connection opened")

    def disconnect(self):
        print("[DB] Connection closed")

    def fetch_rows(self):
        return iter(self.rows)

def db_row_generator(database):
    database.connect()
    try:
        for row in database.fetch_rows():
            yield row
    finally:
        database.disconnect()

db = FakeDatabase([("Alice", 30), ("Bob", 25), ("Carol", 35)])
gen = db_row_generator(db)

print("Row:", next(gen))
gen.close()

**Explanation:** database.connect() runs BEFORE the try block deliberately — there's nothing to clean up if the connection itself never succeeded, so only resources that were actually acquired belong inside the guarded section. The finally clause is guaranteed to run no matter what: whether the loop finishes normally, an exception propagates out, or the generator is closed early — the exact same guarantee a with statement's __exit__ provides. gen.close() works by throwing a GeneratorExit exception into the generator at its current yield point; the generator must not yield again in response, only clean up and end, which is exactly what happens here as disconnect() fires and the generator terminates. (This try/finally-around-a-single-yield pattern is precisely what @contextlib.contextmanager uses internally to implement a context manager from a generator function.)